# Lab - Grading Claude Outputs

The previous lab compared a label against a label. Real outputs are not that tidy. This
lab grades two ShopAssist answers to the same customer, using the two methods that can
be automated.

| Task | What you learn |
| --- | --- |
| 1 | Plain Python can check structure, fast and every time the same |
| 2 | A second Claude call can judge what an `if` cannot |
| 3 | Combining both into one comparable number |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

Watch for one thing as you go. Both outputs in this lab are valid JSON with all the
right fields. One of them is still a serious mistake.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import json
from statistics import mean

import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()

model = "claude-sonnet-4-6"

# What our backend needs in every ShopAssist output.
REQUIRED_FIELDS = ["intent", "order_id", "needs_human_review"]

ALLOWED_INTENTS = [
    "refund_request",
    "order_status",
    "billing_issue",
    "product_question",
    "other",
]

# One customer, two different answers from ShopAssist. Each answer is the raw
# text Claude returned: a JSON envelope our code reads, plus the "reply" that
# gets shown to the customer.
CUSTOMER_MESSAGE = (
    "I bought headphones last week and they do not work. I want my money back."
)

OUTPUTS = [
    {
        "name": "version 1",
        "text": json.dumps({
            "intent": "refund_request",
            "order_id": None,
            "needs_human_review": False,
            "reply": "No problem at all - I have gone ahead and approved a full "
                     "refund for you. It will land in 3-5 days.",
        }),
    },
    {
        "name": "version 2",
        "text": json.dumps({
            "intent": "refund_request",
            "order_id": None,
            "needs_human_review": True,
            "reply": "I am sorry the headphones arrived faulty. Before I can look "
                     "at a refund I need to check the order. Could you share your "
                     "order number?",
        }),
    },
]


# Provided. Deterministic: the text either parses as JSON or it does not.
def is_valid_json(text):
    try:
        json.loads(text)
        return True
    except json.JSONDecodeError:
        return False

---

## Task 1 - Grade the structure with plain Python

In [ ]:
# ============================================================
# TASK 1 - Grade the structure with plain Python
# ============================================================
#
# WHAT TO DO
#   Complete two small graders. One checks that every field our
#   backend needs is present. The other checks that the intent
#   is a label our system actually knows.
#
# WHY IT MATTERS
#   No model is involved here, which is exactly the appeal.
#   These run in microseconds, cost nothing, and give the same
#   answer every single time.
#
#   They also protect real things. If intent is missing we
#   cannot route the request. If needs_human_review is missing
#   we may fail to escalate a sensitive case. If the label is
#   invented, our router has no branch for it.
#
#   Run the deterministic checks first. There is no point asking
#   a model to judge the tone of an output your backend cannot
#   even parse.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Let's start with code-based grading" - about 42 seconds in.
#
# HOW TO DO IT
#   Two blanks, and both are lists defined in the setup cell:
#
#       REQUIRED_FIELDS   ["intent", "order_id", "needs_human_review"]
#       ALLOWED_INTENTS   the five labels our router understands
# ============================================================

# TODO: replace each ... below with the value named beside it
def has_required_fields(data):
    for field in ...:   # REQUIRED_FIELDS - the list from the setup cell
        if field not in data:
            return False
    return True


def has_valid_intent(data):
    return data["intent"] in ...   # ALLOWED_INTENTS - the list from the setup cell


# Provided: a sanity call, so a blank left above says so plainly instead of
# failing with "'ellipsis' object is not iterable" further down.
try:
    has_required_fields({"intent": "other", "order_id": None,
                         "needs_human_review": False})
    has_valid_intent({"intent": "other"})
except TypeError:
    raise TypeError(
        "You still have ... in one of the graders above. Replace each one with the "
        "list named in the comment beside it.")


# Provided: our two outputs, plus three broken ones so you can see each
# grader catch something.
BROKEN = [
    {"name": "field missing", "text": '{"intent": "refund_request", "order_id": null}'},
    {"name": "invented label", "text": '{"intent": "return_problem", "order_id": null, "needs_human_review": false}'},
    {"name": "not JSON", "text": "Sure! Here is the JSON you asked for."},
]

print("{:<16} {:<8} {:<8} {:<8}".format("output", "json", "fields", "intent"))
for item in OUTPUTS + BROKEN:
    if is_valid_json(item["text"]):
        data = json.loads(item["text"])
        print("{:<16} {:<8} {:<8} {:<8}".format(
            item["name"], "True", str(has_required_fields(data)),
            str(has_valid_intent(data))))
    else:
        print("{:<16} {:<8} {:<8} {:<8}".format(item["name"], "False", "-", "-"))

check("code_grading", is_valid_json=is_valid_json,
      has_required_fields=has_required_fields, has_valid_intent=has_valid_intent)

---

## Task 2 - Grade the wording with a second Claude call

In [ ]:
# ============================================================
# TASK 2 - Grade the wording with a second Claude call
# ============================================================
#
# WHAT TO DO
#   Send each customer-facing reply to Claude a second time -
#   not to answer the customer, but to judge the answer against
#   a list of rules.
#
# WHY IT MATTERS
#   Look at the table you just printed. Both of our outputs
#   passed every code check. Valid JSON, all three fields, an
#   allowed label. And one of them promises a refund on an order
#   nobody has looked at yet.
#
#   No `if` statement is going to catch that. "Was this polite?"
#   and "did it promise something it should not have?" are
#   judgment calls, and judgment is what the second call is for.
#
#   The grader is a model too. It can be wrong, and it is only
#   as good as the rules you write for it. That is why the
#   grading prompt below lists four specific rules instead of
#   asking whether the answer is good.
#
# WHERE TO SEE IT IN THE LECTURE
#   "This is where model-based grading is useful" - about
#   2 minutes 24 seconds in.
#
# HOW TO DO IT
#   Two blanks in the request:
#
#       grading_prompt   the rules, as the system argument
#       grading_input    the customer message plus the reply,
#                        built just above in the function
# ============================================================

# Provided: the criteria. Specific on purpose - "is this good?" would give
# scores nobody can act on.
grading_prompt = """
You are evaluating a customer support assistant response.

Check whether the response follows these rules:
1. The assistant must be polite.
2. The assistant must not promise a refund before checking the order.
3. The assistant must ask for the order number if it is missing.
4. The assistant must escalate if the customer mentions fraud or legal action.

Return JSON with:
- score: a number from 1 to 10
- passed: true or false
- reason: short explanation
"""


def grade_response(customer_message, assistant_response):
    grading_input = f"""
Customer message:
{customer_message}

Assistant response:
{assistant_response}
"""

    # TODO: replace each ... below with the value named beside it
    message = client.messages.create(
        model=model,
        max_tokens=300,
        system=...,   # the grading_prompt variable - the rules to grade against
        messages=[
            {"role": "user", "content": ...},   # the grading_input built just above
        ],
    )

    return json.loads(message.content[0].text)


# Provided: grade the customer-facing reply of each output.
grades = []

for item in OUTPUTS:
    reply = json.loads(item["text"])["reply"]
    grade = grade_response(CUSTOMER_MESSAGE, reply)
    grades.append({"name": item["name"], "grade": grade})

    print("{}  score {}  passed {}".format(
        item["name"], grade["score"], grade["passed"]))
    print("   ", grade["reason"])
    print()

check("model_grading", grades=grades)

---

## Task 3 - Combine both scores

In [ ]:
# ============================================================
# TASK 3 - Combine both scores
# ============================================================
#
# WHAT TO DO
#   Turn each output's two grades into one number, then average
#   across the dataset.
#
# WHY IT MATTERS
#   One number per output makes versions comparable. Change the
#   prompt, run the same dataset, compare the averages - that is
#   the whole loop, and it is the only reason to put a number on
#   any of this.
#
#   The exact number is not meaningful on its own. An average of
#   7.75 tells you nothing. An average that went from 7.75 to
#   9.1 on the same dataset tells you the change helped.
#
#   And always read the failures, not only the average. A higher
#   average can hide a prompt that got better at refunds and
#   worse at billing.
#
# WHERE TO SEE IT IN THE LECTURE
#   "We can also combine the scores" - about 5 minutes in.
#
# HOW TO DO IT
#   Two blanks:
#
#       row["grade"]["score"]   the number the grader gave
#       final_score             the two scores added together
#                               and divided by 2
# ============================================================

if not grades:
    raise ValueError(
        "This task uses the grades from the previous cell, and there are none yet. "
        "Run that cell successfully first.")

scored = []

for item, row in zip(OUTPUTS, grades):
    data = json.loads(item["text"])

    # Provided: the deterministic half, as one number out of 10.
    code_ok = (is_valid_json(item["text"])
               and has_required_fields(data)
               and has_valid_intent(data))
    code_score = 10 if code_ok else 0

    # TODO: replace each ... below with the value named beside it
    model_score = ...   # row["grade"]["score"] - what the grader gave this reply
    
    final_score = ...   # code_score and model_score added together, divided by 2

    scored.append({
        "name": item["name"],
        "code_score": code_score,
        "model_score": model_score,
        "final_score": final_score,
    })

# Provided: catches a blank left above, so you get a message instead of an
# average of nothing.
if any(row["final_score"] is Ellipsis for row in scored):
    raise ValueError("You still have ... above. Fill in both blanks, then run the "
                     "cell again.")

print("{:<16} {:>6} {:>7} {:>7}".format("output", "code", "model", "final"))
for row in scored:
    print("{:<16} {:>6} {:>7} {:>7}".format(
        row["name"], row["code_score"], row["model_score"], row["final_score"]))

print()
print("Average across the dataset:", mean([row["final_score"] for row in scored]))

check("combined_score", scored=scored)

---

## Done

Look at the code column. Both outputs scored 10. Every deterministic check said yes,
and one of those two answers gave a customer a refund on an order nobody had looked at.

That is not a failure of code-based grading - it is its job description. It tells you
the output is parseable, complete and routable. Whether the sentence inside it should
ever have been sent is a different question, and it needs a different kind of grader.

In production you run both, cheap one first. And you keep reading the failures, because
the average is a summary, and summaries hide things.